# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nirajmalla123/ML_pipeline/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

- **Unit:** One row represents a client's content item's daily Google Search Console performance.
- **Feature window:** March 1–31, 2026.
- **Outcome window:** April 1–30, 2026.
- **Target:** Predict whether April impressions decline by at least 20% compared with March.

In [ ]:
%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face READ token: ")

con = duckdb.connect()
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"

print("Setup successful!")

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify March data
print(con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS unique_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {MARCH}
""").df())

# Verify April data
print(con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {APRIL}
""").df())

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


- **Features:** March impressions, clicks, CTR, average position, and active days. These are available by March 31.
- **Label:** `decline_20pct`, based on whether April impressions fall below 80% of March impressions.
- **Context:** Date, client ID, content ID, and GSC availability flags. Used for grouping and filtering.
- **Excluded:** April measurements and GA4 fields as model features, because April is future data and GA4 coverage differs.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



required_columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "gsc_data_available"
]

schema = con.sql(f"DESCRIBE SELECT * FROM {MARCH}").df()
available_columns = schema["column_name"].tolist()

print("Missing columns:", [
    col for col in required_columns if col not in available_columns
])


# 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

- **Grain:** Check duplicate client-content-date rows.
- **Counts and dates:** Check row count and date range for March and April.
- **Availability:** Count rows where `gsc_data_available IS TRUE`.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1. Check duplicate rows
print(con.sql(f"""
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS unique_rows
FROM {MARCH}
""").df())

# 2. Check counts and date ranges
print(con.sql(f"""
SELECT 'March' AS month, COUNT(*) AS rows,
       MIN(report_date) AS start_date, MAX(report_date) AS end_date
FROM {MARCH}
UNION ALL
SELECT 'April', COUNT(*), MIN(report_date), MAX(report_date)
FROM {APRIL}
""").df())

# 3. Check available GSC data
print(con.sql(f"""
SELECT COUNT(*) AS total_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_rows
FROM {MARCH}
""").df())


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- GSC data measures search performance, not the cause of changes.
- Missing or uneven data across clients can bias results.
- April outcomes may be affected by seasonality or external events.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_impressions IS NULL
    ) AS missing_impressions,
    COUNT(*) FILTER (
        WHERE gsc_clicks IS NULL
    ) AS missing_clicks
FROM {MARCH}
""").df())


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.